# Converting PyTorch Lightning to Federated Learning

In chapter 1, we have learned how to convert the PyTorch code to a federated learning job with NVIDIA FLARE. In this section, we will learn how to convert the PyTorch Lightning code to a federated learning job with FLARE.

## Basic Concepts

At the heart of FLARE lies the concept of collaboration through "tasks." An FL controller assigns tasks (e.g., training on local data) to one or more FL clients, processes returned results (e.g., model weight updates), and may assign additional tasks based on these results and other factors (e.g., a pre-configured number of training rounds). The clients run executors that can listen for tasks and perform the necessary computations locally, such as model training. This task-based interaction repeats until the experiment's objectives are met.

<img src="./controller_worker_flow.png" alt="Controller-Executor Flow" width="500"/>

## Patch the Lightning trainer

`nvflare.client.lightning.patch(trainer)` adds callbacks that receive the global state before validation/training and send the trained state after `fit()`. Keep the local `LightningModule` and data module. The Client API loop controls federated rounds.

The example uses two disjoint CIFAR-10 shards. Each client uses up to 1,000 images for a short CPU run, split reproducibly into local training and validation sets. Increase that limit in `client.py` for longer experiments.

## Install lesson dependencies

First complete the [course environment setup](../02.0_introduction/introduction.ipynb) to install the matching NVFLARE version and select its Jupyter kernel. The requirements below contain only the additional dependencies for this lesson.

In [ ]:
import os
import sys
from pathlib import Path

# Make notebook shell commands use the selected kernel's environment.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ.get("PATH", "")

lesson = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "convert_torch_lightning_to_fl.ipynb").is_file())
code_dir = lesson / "code"
print("Example code:", code_dir)
%cd {code_dir}

In [ ]:
%cd {code_dir}
!python -m pip install -r requirements.txt
assert _exit_code == 0, "Command failed; see the output above."

## Define the model and client

In [ ]:
print((code_dir / "model.py").read_text())

print((code_dir / "client.py").read_text())

## Configure FedAvg with a recipe

`FedAvgRecipe` provides the server aggregation workflow and PyTorch model persistence. Its initial `LitNet` matches the client's module, and `val_acc` is the metric used for global model selection. There is no custom server controller in this lesson.

In [ ]:
print((code_dir / "job.py").read_text())

## Run and export the recipe

`SimEnv(log_config="progress")` shows workflow progress and a final run summary. Detailed logs remain in the simulation workspace. Each command below uses this notebook's Python and stops if the command fails.

Run `job.py --export` to export the same recipe without training. The exported job can be submitted to a deployed NVIDIA FLARE system.

In [ ]:
%cd {code_dir}
!python job.py
assert _exit_code == 0, "Command failed; see the output above."

In [ ]:
%cd {code_dir}
!python job.py --export --export-dir /tmp/nvflare/chapter2/exports/lightning
assert _exit_code == 0, "Command failed; see the output above."

The run summary reports completion and the result location under `/tmp/nvflare/chapter2/lightning/lightning_fedavg`. Each round validates the received global model before local training.

Continue with [machine learning examples](../02.4_convert_machine_learning_to_federated_learning/convert_ml_to_fl.ipynb).